# AFS Analyze

Generates a consulting briefing for a loaded organization: ratio table, trend deltas, and `COMMON.FINDINGS`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from afs.cortex_llm import init as llm_init
from afs.snowflake_io import cursor_from_session
from afs.insights import compute_ratios, compute_trends, synthesize_findings, write_findings

llm_init(session)
_ctx = cursor_from_session(session, commit_on_exit=False)
cur = _ctx.__enter__()

In [ ]:
# Show available organizations
import pandas as pd

orgs = session.sql("""
    SELECT ORG_CODE, LEGAL_NAME, SECTOR, HQ_STATE
      FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY
     ORDER BY LEGAL_NAME
""").to_pandas()

print(f'{len(orgs)} organization(s) in registry:')
orgs

In [ ]:
orgs_list = session.sql("""
    SELECT DISTINCT r.ORG_ID, r.ORG_CODE, r.LEGAL_NAME
      FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY r
      JOIN AUDITED_FINANCIALS.COMMON.FILINGS f ON f.ORG_ID = r.ORG_ID
     ORDER BY r.LEGAL_NAME
""").collect()

print(f'{len(orgs_list)} organization(s) with filings:')
for o in orgs_list:
    print(f'  {o[1]} — {o[2]}')

In [ ]:
import pandas as pd

all_results = []

for org_row in orgs_list:
    org_id, org_code, legal_name = org_row[0], org_row[1], org_row[2]
    print(f'\n=== {legal_name} ({org_code}) ===')

    ratios = compute_ratios(cur, org_id)
    trends = compute_trends(ratios)
    years = ratios['years']
    print(f'  FY years with data: {years}')

    ratio_rows = []
    for metric in next(iter(ratios['ratios_by_fy'].values()), {}):
        row_data = {'metric': metric}
        for yr in years:
            v = ratios['ratios_by_fy'].get(yr, {}).get(metric)
            row_data[yr] = f'{v:,.2f}' if v is not None else '—'
        ratio_rows.append(row_data)
    df_ratios = pd.DataFrame(ratio_rows).set_index('metric')
    print(df_ratios.to_string())

    all_results.append({
        'org_id': org_id, 'org_code': org_code, 'legal_name': legal_name,
        'ratios': ratios, 'trends': trends,
    })

In [ ]:
total_written = 0

for res in all_results:
    org_id = res['org_id']
    org_code = res['org_code']
    ratios = res['ratios']
    trends = res['trends']
    print(f'\n=== Findings: {res["legal_name"]} ===')

    cur.execute(
        "DELETE FROM COMMON.FINDINGS WHERE ORG_ID = %s", (org_id,)
    )
    cur.connection.commit()

    cur.execute("""
        SELECT FY_LABEL, FILING_ID
          FROM AUDITED_FINANCIALS.COMMON.FILINGS
         WHERE ORG_ID = %s
         GROUP BY FY_LABEL, FILING_ID
         ORDER BY FY_LABEL
    """, (org_id,))
    fy_filings = cur.fetchall()

    for fy_label, filing_id in fy_filings:
        print(f'  {fy_label} (filing {filing_id[:12]}...) ', end='')
        try:
            findings = synthesize_findings(ratios, trends, org_code)
            n = write_findings(cur, org_id, filing_id, fy_label, findings)
            cur.connection.commit()
            total_written += n
            print(f'→ {n} finding(s)')
        except Exception as e:
            print(f'→ ERROR: {e}')

print(f'\nTotal findings written: {total_written}')

In [ ]:
from snowflake.snowpark.functions import col

df_findings = session.table('AUDITED_FINANCIALS.COMMON.FINDINGS').select(
    'FY_LABEL', 'SEVERITY', 'CATEGORY', 'TITLE',
    'EST_IMPACT_LOW', 'EST_IMPACT_HIGH', 'NARRATIVE', 'PLAYBOOK_HINT'
).order_by(col('FY_LABEL'), col('SEVERITY')).to_pandas()

print(f'{len(df_findings)} total finding(s) across all orgs/years:\n')
for _, f in df_findings.iterrows():
    lo = f.get('EST_IMPACT_LOW')
    hi = f.get('EST_IMPACT_HIGH')
    rng = f'${lo:,.0f}\u2013${hi:,.0f}' if lo and hi else '\u2014'
    print(f"[{f['SEVERITY'].upper()}] {f['FY_LABEL']} | {f['TITLE']}  ({f['CATEGORY']}, {rng})")
    print(f"  Playbook: {f['PLAYBOOK_HINT'] or '\u2014'}")
    print(f"  {f['NARRATIVE']}\n")

In [ ]:
_ctx.__exit__(None, None, None)

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()